# 09 · Experiment Design Appendix
**Purpose:** Convert every recommendation in the action matrix into a testable hypothesis with a statistically valid experimental design.

> **Principle:** A recommendation without a measurement plan is a guess. A measurement plan without a sample size calculation is wishful thinking.

This notebook documents the methodology choice (A/B vs quasi-experiment vs Bayesian), the required sample size, minimum duration, win condition, and guardrail metric for each proposed intervention.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from scipy.stats import norm as norm_dist, chi2
import warnings; warnings.filterwarnings('ignore')

def sample_size(baseline, mde, alpha=0.05, power=0.80):
    p1=min(float(baseline),.9999); p2=min(p1*(1+mde),.9999)
    es=abs(p2-p1)/np.sqrt((p1*(1-p1)+p2*(1-p2))/2)
    if es < 1e-9: return 99999
    z = (norm_dist.ppf(1-alpha/2) + norm_dist.ppf(power))/es
    return int(np.ceil(z**2))

experiments = [
    {'name':'Pro $79→$40 reprice','baseline':0.18,'mde':0.15,'vol_wk':45,'method':'A/B'},
    {'name':'Enterprise $299→$500','baseline':0.31,'mde':-0.10,'vol_wk':10,'method':'Bayesian sequential'},
    {'name':'Starter onboarding','baseline':0.16,'mde':0.25,'vol_wk':55,'method':'A/B'},
    {'name':'LinkedIn budget shift','baseline':3.90,'mde':0.20,'vol_wk':12,'method':'DiD'},
    {'name':'Annual billing offer','baseline':0.26,'mde':0.25,'vol_wk':30,'method':'A/B'},  # baseline: 26% retention improvement vs monthly observed in data
]
for e in experiments:
    n = sample_size(e['baseline'], abs(e['mde']))
    e['n_arm'] = n; e['total_n'] = 2*n
    e['weeks'] = max(int(np.ceil(2*n/e['vol_wk'])),4)
    e['feasible'] = '✓' if e['weeks']<=52 else '✗ Infeasible as A/B'
df = pd.DataFrame(experiments)
print(df[['name','baseline','mde','n_arm','weeks','method','feasible']].to_string(index=False))

In [ ]:
# Power curves — visualise tradeoff between MDE and sample size
fig, axes = plt.subplots(1,2,figsize=(13,5))
mde_range = np.linspace(0.05, 0.50, 50)
for base, label, color in [(0.16,'Starter (base=16%)','#e74c3c'),
                           (0.18,'Pro (base=18%)','#f39c12'),
                           (0.31,'Enterprise (base=31%)','#27ae60')]:
    ns = [sample_size(base, m) for m in mde_range]
    axes[0].plot(mde_range*100, ns, label=label, color=color, linewidth=2)
axes[0].axhline(y=500,  color='gray', linestyle='--', alpha=0.5, label='n=500 per arm')
axes[0].axhline(y=2000, color='gray', linestyle=':', alpha=0.5, label='n=2000 per arm')
axes[0].set_xlabel('Minimum Detectable Effect (%)'); axes[0].set_ylabel('Sample size per arm')
axes[0].set_title('Sample Size vs Effect Size by Experiment', fontweight='bold')
axes[0].legend(fontsize=9); axes[0].set_ylim(0,5000)

# Duration curves at different weekly volumes
vol_range = np.arange(5, 100, 5)
for exp in [{'name':'Pro reprice','base':0.18,'mde':0.15,'color':'#f39c12'},
            {'name':'Starter onboard','base':0.16,'mde':0.25,'color':'#e74c3c'},
            {'name':'Annual billing','base':0.16,'mde':0.30,'color':'#9b59b6'}]:
    n = sample_size(exp['base'], exp['mde'])
    weeks = [max(int(np.ceil(2*n/v)),4) for v in vol_range]
    axes[1].plot(vol_range, weeks, label=exp['name'], color=exp['color'], linewidth=2)
axes[1].axhline(y=52, color='red', linestyle='--', alpha=0.7, label='52-week limit')
axes[1].set_xlabel('Weekly signup volume'); axes[1].set_ylabel('Test duration (weeks)')
axes[1].set_title('Test Duration vs Traffic Volume', fontweight='bold')
axes[1].legend(fontsize=9); axes[1].set_ylim(0,120)
plt.suptitle('Experiment Power Analysis', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout(); plt.savefig('../reports/experiment_power_curves.png', dpi=150); plt.show()

## Methodology Decision Tree

```
Is weekly eligible volume > 50?
├── YES → Can you randomise at the individual level?
│         ├── YES → Run standard A/B test (α=0.05, power=0.80)
│         └── NO  → Run Difference-in-Differences (pre/post periods)
└── NO  → Is the test reversible (can you undo the change)?
          ├── YES → Run Bayesian sequential test (stop early if posterior clear)
          └── NO  → Do not test — run staged rollout with monitoring only
```

**Enterprise repricing falls into: volume < 50, reversible → Bayesian sequential.**  
**LinkedIn budget shift falls into: cannot randomise individuals → Difference-in-Differences.**  
These methodology choices are not optional — the wrong method produces a result you cannot trust.

### Annual billing experiment — design note

The annual billing experiment (Experiment 5) is not simply testing whether to offer annual plans — the company already does. The specific hypothesis being tested is whether **gating the annual offer behind an onboarding completion threshold** improves retention vs showing it at signup unconditionally.

The observational data shows annual customers churn more (adverse selection). The RD design in notebook 10 shows that when customers are genuinely committed to annual billing, retention improves. The experiment tests the mechanic that would separate these two populations: requiring customers to hit a product activation milestone before the annual offer appears. Baseline = current 26% 12-month retention rate for annual signups. Win condition = 35%+ retention at 12 months (MDE = +9pp).

## Analyst Sign-Off

Every recommendation in the action matrix now has:
- ✓ A falsifiable hypothesis
- ✓ A primary metric with baseline and MDE
- ✓ A statistically valid methodology choice
- ✓ A sample size calculation based on actual weekly volume
- ✓ A minimum duration (never declare early)
- ✓ A win condition (binary — either met or not)
- ✓ A guardrail metric (what must NOT get worse)

This transforms the action matrix from a to-do list into a product roadmap with built-in measurement.